In [1]:
import argparse
import json
import os
import tempfile
import numpy as np
import subprocess
from tqdm import tqdm
import pandas as pd
import datetime
import time
import gc
import random
import sys
import shutil
import glob

sys.path.append('/home/agustin/phd/synthesis')
sys.path.append('/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/test4_finetune_brainst_diffusion_model/training/networks_declaration')


# pytorch
import torch
from torch.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, Sampler
import torch.distributed as dist
# data loader
from sklearn.preprocessing import MinMaxScaler

# mine
import utils.nifti_functions as nfc
import utils.util as util
import utils.functions as fc
import utils.util_freesurfer_segmentation as ufs
import utils.gpu_selector as gpu_selector
import data_loaders.load_dataset as load_dataset
import utils.data_normalization as data_normalization


# monai
from monai.bundle import ConfigParser

import  networks_declaration.diffusion_model_unet_maisi_mask_att as diffusion_model_unet_maisi
import  networks_declaration.volumne_encoder as volumne_encoder


# import attention_controller as attention_controller
from networks_declaration.rectified_flow import RFlowScheduler
# from monai.networks.schedulers.rectified_flow import RFlowScheduler
from monai.networks.schedulers.ddpm import DDPMPredictionType
# images
from PIL import Image

sys.path.append('/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/utils')
from autoencoder_declaration import AutoencoderPrediction

# device_name = f"cuda:{gpu_selector.get_least_used_gpu()}"
device_name = f"cuda:1"
device = torch.device(device_name)


def set_seed(seed: int):
    # random.seed(seed)  # Semilla para Python
    np.random.seed(seed)  # Semilla para NumPy
    torch.manual_seed(seed)  # Semilla para PyTorch en CPU
    torch.cuda.manual_seed(seed)  # Semilla para PyTorch en GPU
    torch.cuda.manual_seed_all(seed)  # Semilla para todas las GPUs
    torch.backends.cudnn.deterministic = True  # Garantizar reproducibilidad en CNNs
    torch.backends.cudnn.benchmark = False  # Desactivar optimización no determinista


In [4]:

def instantiate_unconditioned_models(device, noise_scheduler_type="rflow"):

    networks_config =  {
        
        "diffusion_unet_def": {
            "_target_": "monai.apps.generation.maisi.networks.diffusion_model_unet_maisi.DiffusionModelUNetMaisi",
            "spatial_dims": 3,
            "in_channels": 4,
            "out_channels": 4,
            "num_res_blocks": 2,
            "num_channels": [
                # 64,
                # 128,
                # 256,
                # 512

                32,
                64,
                128,
            ],
            "self_attention_levels": [
                False,
                False,
                # True,
                True
            ],

            "num_self_head_channels": [
                0,
                0,
                # 32,
                32
            ],
            "cross_attention_levels": [
                False,
                # False,
                False,
                False
            ],
            "num_cross_head_channels": [
                0,
                # 0,
                0,
                0
            ],

            "use_flash_attention": True,
            "with_conditioning": False,
            "cross_attention_dim": None,
            "transformer_num_layers": 1, # number of transformer blocks
            "upcast_attention": True,

        },

        "conditions_model": { # this is volumetric conditioning
            "num_conditions": 18,  # number of conditions
            "embed_dim": 512,  # same as the cross_attention_dim in the unet
            "hidden_dim": [128, 256],  # half of the embedding dimension
            "use_self_attention": False,  # whether to use self-attention
            "n_heads": 8,  # number of attention heads
            "n_att_layers": 1,  # number of layers in the MLP
            "use_gelu": True, # whether to use gelu or relu
        },



        "noise_scheduler": {
            "_target_": "monai.networks.schedulers.DDIMScheduler", # faster scheduler
            "beta_start": 0.0015,
            "beta_end": 0.0205,
            "num_train_timesteps": 1000,
            "schedule": "scaled_linear_beta",
            "clip_sample": False
        },


        "noise_scheduler_rf": {
        "_target_": "monai.networks.schedulers.rectified_flow.RFlowScheduler",
        "num_train_timesteps": 1000,
        "use_discrete_timesteps": False,
        "use_timestep_transform": True,
        "sample_method": "uniform",
        "scale":1.4
        },


    }


    # instantiate model
    parser = ConfigParser(networks_config)
    parser.parse(True)

    args = fc.dict_to_args(networks_config, deep_conversion=True)

    # unet
    unet = diffusion_model_unet_maisi.DiffusionModelUNetMaisi(spatial_dims = args.diffusion_unet_def.spatial_dims,
                                                in_channels = args.diffusion_unet_def.in_channels,
                                                out_channels = args.diffusion_unet_def.out_channels,
                                                num_res_blocks = args.diffusion_unet_def.num_res_blocks,
                                                num_channels = args.diffusion_unet_def.num_channels,
                                                # attention_levels = args.diffusion_unet_def.attention_levels,
                                                self_attention_levels = args.diffusion_unet_def.self_attention_levels,
                                                cross_attention_levels = args.diffusion_unet_def.cross_attention_levels,
                                                # num_head_channels = args.diffusion_unet_def.num_head_channels,
                                                num_self_head_channels = args.diffusion_unet_def.num_self_head_channels,
                                                num_cross_head_channels = args.diffusion_unet_def.num_cross_head_channels,
                                                with_conditioning = args.diffusion_unet_def.with_conditioning,
                                                transformer_num_layers = args.diffusion_unet_def.transformer_num_layers,
                                                cross_attention_dim = args.diffusion_unet_def.cross_attention_dim,
                                                upcast_attention = args.diffusion_unet_def.upcast_attention,
                                                use_flash_attention = args.diffusion_unet_def.use_flash_attention,
                                                include_top_region_index_input=False,
                                                include_bottom_region_index_input=False,
                                                include_spacing_input=False
                                                )
    

    # ConditionTokens
    conditions_model = volumne_encoder.ConditionTokens(num_conditions=args.conditions_model.num_conditions, 
                                    embed_dim=args.conditions_model.embed_dim,
                                    hidden_dim=args.conditions_model.hidden_dim,
                                    use_self_attention=args.conditions_model.use_self_attention, 
                                    n_heads=args.conditions_model.n_heads, 
                                    n_layers=args.conditions_model.n_att_layers,
                                    use_gelu=args.conditions_model.use_gelu
                                    )

    # noise scheduler
    if noise_scheduler_type == "ddim":
        noise_scheduler = parser.get_parsed_content("noise_scheduler", instantiate=True)
        noise_scheduler.set_timesteps(num_inference_steps=50)
    elif noise_scheduler_type == "rflow":
        # noise_scheduler = parser.get_parsed_content("noise_scheduler_rf", instantiate=True)
        noise_scheduler = RFlowScheduler(num_train_timesteps=args.noise_scheduler_rf.num_train_timesteps,
                                        use_discrete_timesteps=args.noise_scheduler_rf.use_discrete_timesteps,
                                        use_timestep_transform=args.noise_scheduler_rf.use_timestep_transform,
                                        sample_method=args.noise_scheduler_rf.sample_method,
                                        scale=args.noise_scheduler_rf.scale)
        noise_scheduler.set_timesteps(num_inference_steps=30,
                                    input_img_size_numel=torch.prod(torch.tensor((48,64,48))))

    # autoencoder (just for validation)
    # autoencoder = parser.get_parsed_content("autoencoder_def").to(device)
    autoencoder_chekpoint_path = "/home/agustin/phd/synthesis/tests/D3/maisi/understanding_vae/vae_weights/autoencoder_epoch273.pt"
    # checkpoint_autoencoder = torch.load(autoencoder_chekpoint_path, weights_only=True, map_location=device)
    # autoencoder.load_state_dict(checkpoint_autoencoder)
    # autoencoder.eval()
    
    autoencoder = AutoencoderPrediction(autoencoder_chekpoint_path, device, half=True)
    


    return {"unet": unet, 
              "autoencoder": autoencoder, 
              "conditions_model": conditions_model,
              "noise_scheduler": noise_scheduler,
              "networks_config": args}



models_dict = instantiate_unconditioned_models(device, noise_scheduler_type="rflow")
unet = models_dict["unet"]
conditions_model = models_dict["conditions_model"]
autoencoder = models_dict["autoencoder"]
networks_config = models_dict["networks_config"]
noise_scheduler = models_dict["noise_scheduler"]


unet.to(device)
unet.train()
a =2
unet
# print("number of parameters in unet: ", sum(p.numel() for p in unet.parameters()))

DiffusionModelUNetMaisi(
  (conv_in): Convolution(
    (conv): Conv3d(4, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  )
  (time_embed): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (down_blocks): ModuleList(
    (0): DownBlock(
      (resnets): ModuleList(
        (0-1): 2 x DiffusionUNetResnetBlock(
          (norm1): GroupNorm(32, 32, eps=1e-06, affine=True)
          (nonlinearity): SiLU()
          (conv1): Convolution(
            (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          )
          (time_emb_proj): Linear(in_features=128, out_features=32, bias=True)
          (norm2): GroupNorm(32, 32, eps=1e-06, affine=True)
          (conv2): Convolution(
            (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          )
          (skip_connection): Identity()
        )
   

In [21]:

print("trainable params:", sum(p.requires_grad for p in unet.parameters()))


# for name, param in unet.named_parameters():
#     param.requires_grad = False
    
# for p in unet.out.parameters():
#     p.requires_grad = True

# for p in unet.up_blocks[3].parameters():
#     p.requires_grad = True

# for p in unet.middle_block.parameters():
#     p.requires_grad = True

# for p in unet.down_blocks[3].parameters():
#     p.requires_grad = True

def freeze_module(m):
    for p in m.parameters():
        p.requires_grad = False

def unfreeze_module(m):
    for p in m.parameters():
        p.requires_grad = True

freeze_module(unet)

unfreeze_module(unet.out)
unfreeze_module(unet.up_blocks[3])
unfreeze_module(unet.middle_block)
unfreeze_module(unet.down_blocks[3])

print("trainable params:", sum(p.requires_grad for p in unet.parameters()))


trainable params: 748
trainable params: 217
